In [2]:
!pip install tensorflow

   ---------------------------------------- 0.0/350.9 MB ? eta -:--:--
    --------------------------------------- 6.3/350.9 MB 42.8 MB/s eta 0:00:09
    --------------------------------------- 6.3/350.9 MB 42.8 MB/s eta 0:00:09
    --------------------------------------- 7.1/350.9 MB 13.2 MB/s eta 0:00:27
   -- ------------------------------------- 19.1/350.9 MB 23.7 MB/s eta 0:00:15
   --- ------------------------------------ 34.3/350.9 MB 34.1 MB/s eta 0:00:10
   ----- ---------------------------------- 50.1/350.9 MB 40.9 MB/s eta 0:00:08
   ------ --------------------------------- 59.0/350.9 MB 40.9 MB/s eta 0:00:08
   -------- ------------------------------- 71.0/350.9 MB 42.7 MB/s eta 0:00:07
   -------- ------------------------------- 75.8/350.9 MB 40.3 MB/s eta 0:00:07
   --------- ------------------------------ 83.6/350.9 MB 40.1 MB/s eta 0:00:07
   ---------- ----------------------------- 91.8/350.9 MB 39.8 MB/s eta 0:00:07
   ----------- ---------------------------- 99.6/350

### Imports

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping


c:\Users\Olive\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.1) or chardet (7.3.0)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


In [4]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

cols = [
    "age","workclass","fnlwgt","education","education-num",
    "marital-status","occupation","relationship","race",
    "sex","capital-gain","capital-loss","hours-per-week",
    "native-country","income"
]

df = pd.read_csv(url, names=cols)


### Cleaning

In [5]:
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

In [6]:
# remove missing '?'
df = df.replace('?', np.nan)
df = df.dropna()

In [7]:
df["income"] = df["income"].map({"<=50K": 0, ">50K": 1})

### Feature Engineering

In [8]:
df["hours_per_age"] = df["hours-per-week"] / df["age"]

In [9]:
df = pd.get_dummies(df, drop_first=True)

X = df.drop("income", axis=1)
y = df["income"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Scale

In [11]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Model 1 - Deep Learning

In [12]:
model = Sequential()

model.add(Dense(128, activation='relu', input_dim=X_train.shape[1]))
model.add(BatchNormalization())
model.add(Dropout(0.4))

c:\Users\Olive\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))

In [14]:
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

In [15]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [16]:
# Early stopping (diferencial)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [17]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
604/604 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8081 - loss: 0.4092 - val_accuracy: 0.8375 - val_loss: 0.3494
Epoch 2/50
604/604 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8276 - loss: 0.3634 - val_accuracy: 0.8380 - val_loss: 0.3471
Epoch 3/50
604/604 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8324 - loss: 0.3536 - val_accuracy: 0.8438 - val_loss: 0.3393
Epoch 4/50
604/604 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8407 - loss: 0.3444 - val_accuracy: 0.8450 - val_loss: 0.3374
Epoch 5/50
604/604 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8413 - loss: 0.3386 - val_accuracy: 0.8465 - val_loss: 0.3346
Epoch 6/50
604/604 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8452 - loss: 0.3333 - val_accuracy: 0.8494 - val_loss: 0.3341
Epoch 7/50
604/604 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8434 - loss: 0.3308 - val_accuracy: 0.8465 - val_loss: 0.3309
Epoch 8/50
604/604 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8440 - loss: 0.3280 - val_accuracy: 0.

### Evaluation NN

In [18]:
y_pred_nn = (model.predict(X_test) > 0.5).astype(int)

acc_nn = accuracy_score(y_test, y_pred_nn)

print("\n=== Neural Network ===")
print(f"Acurácia: {acc_nn:.4f}")
print(classification_report(y_test, y_pred_nn))

189/189 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step

=== Neural Network ===
Acurácia: 0.8548
              precision    recall  f1-score   support

           0       0.89      0.92      0.90      4503
           1       0.74      0.66      0.70      1530

    accuracy                           0.85      6033
   macro avg       0.81      0.79      0.80      6033
weighted avg       0.85      0.85      0.85      6033



# Model 2 - Random Forest

In [19]:
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [20]:
y_pred_rf = rf.predict(X_test)